In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

import optuna
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

from config.paths import config

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

In [8]:
processed_data = joblib.load(config.processed_data_dir / "feature_engineered_data.pkl")
X_train = processed_data['X_train']
X_test = processed_data['X_test']
y_train = processed_data['y_train']
y_test = processed_data['y_test']
feature_names = processed_data['feature_names']

with open(config.reports_dir / 'fast_baseline_models_results.json', 'r') as f:
    baseline_results = json.load(f)

print(f"Training data: {X_train.shape}")
print(f"Testing data: {X_test.shape}")

Training data: (156156, 82)
Testing data: (39040, 82)


In [9]:
performance_data = []
for model_data in baseline_results['performance_summary']:
    accuracy_str = model_data['Accuracy'].split(' ± ')[0]
    roc_auc_str = model_data['ROC-AUC'].split(' ± ')[0]
    training_time_str = model_data['Training Time (s)']

    performance_data.append({
        'model': model_data['Model'],
        'accuracy': float(accuracy_str),
        'roc_auc': float(roc_auc_str),
        'training_time': float(training_time_str)
    })

performance_df = pd.DataFrame(performance_data)
performance_df['overall_score'] = (performance_df['accuracy'] * 0.4 +
                                  performance_df['roc_auc'] * 0.4 +
                                  (1 / performance_df['training_time']) * 0.2)

all_models = performance_df['model'].tolist()

print("ALL MODELS AVAILABLE FOR TUNING:")
for i, (idx, row) in enumerate(performance_df.iterrows()):
    print(f"{i+1}. {row['model']}: Accuracy: {row['accuracy']:.4f}, ROC-AUC: {row['roc_auc']:.4f}")

ALL MODELS AVAILABLE FOR TUNING:
1. Logistic Regression: Accuracy: 0.9741, ROC-AUC: 0.9960
2. Random Forest: Accuracy: 0.9759, ROC-AUC: 0.9962
3. XGBoost: Accuracy: 0.9832, ROC-AUC: 0.9986
4. LightGBM: Accuracy: 0.9832, ROC-AUC: 0.9986
5. Linear SVM: Accuracy: 0.9729, ROC-AUC: 0.9961
6. KNN: Accuracy: 0.9339, ROC-AUC: 0.9652
